## 1. Set keywords to search for

Translate english keywords into Japanese to search for in the manifestos

Deflation = デフレーション, デフレ \
Nuclear Power = 原子力 \
China = 中国 \
Constitution = 憲法 \
Migration = 移民 (Uncommon), 外国人労働者 ("Foreign Worker", More Common)


In [2]:
import pandas as pd
import spacy
from collections import Counter

nlp = spacy.load("ja_core_news_sm")

# Make sure text length isnt too long for spacy
def chunk_text(text, max_bytes=40000):
    encoded = text.encode("utf-8")
    chunks = []
    while encoded:
        chunk = encoded[:max_bytes]
        # avoid splitting mid-character
        chunk = chunk.decode("utf-8", errors="ignore")
        chunks.append(chunk)
        encoded = encoded[len(chunk.encode("utf-8")):]
    return chunks

# Exclude common particles etc.
excluded_words = {"の", "に", "は", "を", "た", "が", "で", "て", "と", "し", "ます", "する", "な", "や", "も", "へ"}

# Tokenize manifesto text
def tokenize(csv_path):
    df = pd.read_csv(csv_path)
    all_tokens = []
    
    for text in df["text"]:
        for chunk in chunk_text(str(text)):
            doc = nlp(chunk)
            tokens = [token.text for token in doc 
                      if not token.is_space 
                      and not token.is_punct
                      and token.text not in excluded_words]
            all_tokens.extend(tokens)
    
    token_df = pd.DataFrame(all_tokens, columns=["word"])
    word_counts_df = token_df["word"].value_counts().reset_index()
    word_counts_df.columns = ["word", "count"]
    
    return word_counts_df

word_counts_df = tokenize("../../project/data/clean/Japan_sorted_manifestos.csv")

In [3]:
word_counts_df.head(10)

,word,count
0,的,566
1,化,530
2,者,444
3,等,409
4,法,370
5,支援,364
6,が,350
7,ため,331
8,で,314
9,など,313


In [4]:
# Search for count of a specific word
def wordsearch(keyword):
    count = word_counts_df.loc[word_counts_df['word'] == keyword, 'count'].item()
    return count

wordsearch('的')

566

In [7]:
import re

phrases = ["経済の再生", "新しい資本主義", "物価高対策", "全世代型社会保障",
           "政治とカネ", "国益を守る", "日本第一"]

def phrase_counts(csv_path, phrases):
    df = pd.read_csv(csv_path)
    df["text"] = df["text"].astype(str)

    results = []
    for phrase in phrases:
        df["count"] = df["text"].str.count(re.escape(phrase))
        total = df["count"].sum()
        by_party = df.groupby("partyname")["count"].sum()
        results.append({
            "phrase": phrase,
            "total": total,
            **by_party.to_dict()
        })

    return pd.DataFrame(results).fillna(0)

results_df = phrase_counts("../../project/data/clean/Japan_sorted_manifestos.csv", phrases)
results_df

,phrase,total,Constitutional Democratic Party of Japan,Japan Restoration Party,Japanese Communist Party,Liberal Democratic Party,New Clean Government Party,Party of Hope,Social Democratic Party
0,経済の再生,1,0,0,1,0,0,0,0
1,新しい資本主義,0,0,0,0,0,0,0,0
2,物価高対策,0,0,0,0,0,0,0,0
3,全世代型社会保障,2,0,0,0,2,0,0,0
4,政治とカネ,5,0,1,3,0,0,0,1
5,国益を守る,1,0,0,0,1,0,0,0
6,日本第一,0,0,0,0,0,0,0,0


Phrases initially tried that we determined above


In [14]:
phrases = ["デフレ", "デフレーション", "原子力", "憲法", "移民", "外国人労働者","国家安全保障", "安全保障"]

def phrase_counts_by_date(csv_path, phrases, start_year=2001, end_year=2021):
    df = pd.read_csv(csv_path)
    df["text"] = df["text"].astype(str)

    df["year"] = pd.to_datetime(df["date"]).dt.year
    df = df[(df["year"] >= start_year) & (df["year"] <= end_year)]

    results = []
    for phrase in phrases:
        df["count"] = df["text"].str.count(re.escape(phrase))
        total = df["count"].sum()
        by_party = df.groupby("partyname")["count"].sum()
        results.append({
            "phrase": phrase,
            "total": total,
            **by_party.to_dict()
        })

    return pd.DataFrame(results).fillna(0)

results_df = phrase_counts_by_date(
    "../../project/data/clean/Japan_sorted_manifestos.csv",
    phrases,
    start_year=2001,
    end_year=2021
)
results_df

,phrase,total,Constitutional Democratic Party of Japan,Japan Restoration Party,Japanese Communist Party,Liberal Democratic Party,New Clean Government Party,Party of Hope,Social Democratic Party
0,デフレ,1,0,0,0,0,0,0,1
1,デフレーション,0,0,0,0,0,0,0,0
2,原子力,23,0,5,1,6,1,3,7
3,憲法,153,7,20,49,13,13,20,31
4,移民,0,0,0,0,0,0,0,0
5,外国人労働者,2,0,0,0,1,0,1,0
6,国家安全保障,0,0,0,0,0,0,0,0
7,安全保障,43,3,12,2,3,8,9,6


Pulling phrases from manifestos in 2001-2021, to correlate with news analysis notebook

In [18]:
from collections import Counter

def top_words_by_party(csv_path, top_n=10, start_year=2001, end_year=2021):
    df = pd.read_csv(csv_path)
    df["year"] = pd.to_datetime(df["date"]).dt.year
    df = df[(df["year"] >= start_year) & (df["year"] <= end_year)]

    results = {}

    for party, group in df.groupby("partyname"):
        tokens = []
        for text in group["text"]:
            for chunk in chunk_text(str(text)):
                doc = nlp(chunk)
                tokens.extend([
                    token.text for token in doc
                    if not token.is_space
                    and not token.is_punct
                    and token.text not in excluded_words
                ])
        results[party] = Counter(tokens).most_common(top_n)

    return results

party_top_words = top_words_by_party(
    "../../project/data/clean/Japan_sorted_manifestos.csv",
    top_n=10,
    start_year=2001,
    end_year=2021
)

for party, words in party_top_words.items():
    print(f"\n{party}:")
    for word, count in words:
        print(f"  {word}: {count}")



Constitutional Democratic Party of Japan:
  社会: 17
  的: 13
  さ: 12
  実現: 12
  原発: 12
  です: 11
  法: 11
  支援: 10
  化: 10
  主義: 9

Japan Restoration Party:
  法: 134
  化: 123
  案: 112
  平成: 107
  改革: 100
  等: 83
  年: 72
  で: 71
  大阪: 63
  的: 60

Japanese Communist Party:
  日本: 173
  です: 164
  い: 152
  税: 149
  こと: 143
  企業: 139
  党: 136
  など: 136
  的: 131
  円: 131

Liberal Democratic Party:
  的: 125
  化: 120
  等: 117
  で: 103
  が: 95
  る: 89
  など: 84
  者: 81
  国: 77
  す: 75

New Clean Government Party:
  等: 161
  など: 155
  が: 147
  支援: 147
  化: 136
  者: 126
  的: 122
  ため: 116
  推進: 109
  で: 81

Party of Hope:
  で: 59
  が: 57
  など: 53
  ゼロ: 36
  より: 32
  社会: 30
  こと: 28
  化: 28
  的: 27
  国民: 21

Social Democratic Party:
  的: 88
  法: 84
  者: 79
  ○: 70
  支援: 70
  など: 67
  地域: 65
  制度: 61
  化: 55
  強化: 49


words common to every party, with each party's count 

In [24]:
#words common to every party, with each party's count 
from collections import Counter
from itertools import combinations

def get_party_word_counts(csv_path, start_year=2001, end_year=2021):
    df = pd.read_csv(csv_path)
    df["year"] = pd.to_datetime(df["date"]).dt.year
    df = df[(df["year"] >= start_year) & (df["year"] <= end_year)]

    party_counts = {}
    for party, group in df.groupby("partyname"):
        tokens = []
        for text in group["text"]:
            for chunk in chunk_text(str(text)):
                doc = nlp(chunk)
                tokens.extend([
                    token.text for token in doc
                    if not token.is_space
                    and not token.is_punct
                    and token.text not in excluded_words
                ])
        party_counts[party] = Counter(tokens)

    return party_counts

party_counts = get_party_word_counts(
    "../../project/data/clean/Japan_sorted_manifestos.csv",
    start_year=2001,
    end_year=2021
)

party_word_sets = {party: set(counter) for party, counter in party_counts.items()}
common_to_all = set.intersection(*party_word_sets.values())

common_all_df = pd.DataFrame([
    {"word": word, **{party: party_counts[party][word] for party in party_counts}}
    for word in common_to_all
]).sort_values(by=list(party_counts.keys())[0], ascending=False)

common_all_df



,word,Constitutional Democratic Party of Japan,Japan Restoration Party,Japanese Communist Party,Liberal Democratic Party,New Clean Government Party,Party of Hope,Social Democratic Party
13,社会,17,31,77,39,47,30,45
97,的,13,60,131,125,122,27,88
161,実現,12,45,27,37,47,20,42
204,さ,12,14,106,27,40,13,24
87,原発,12,12,79,2,6,17,29
...,...,...,...,...,...,...,...,...
86,5,1,19,22,7,10,3,1
85,中小,1,2,40,16,14,7,14
82,月,1,44,18,9,3,3,13
80,法律,1,11,10,2,2,1,2


In [25]:
from collections import Counter

def get_party_word_counts(csv_path, start_year=2001, end_year=2021):
    df = pd.read_csv(csv_path)
    df["year"] = pd.to_datetime(df["date"]).dt.year
    df = df[(df["year"] >= start_year) & (df["year"] <= end_year)]

    party_counts = {}
    for party, group in df.groupby("partyname"):
        tokens = []
        for text in group["text"]:
            for chunk in chunk_text(str(text)):
                doc = nlp(chunk)
                tokens.extend([
                    token.text for token in doc
                    if not token.is_space
                    and not token.is_punct
                    and token.text not in excluded_words
                ])
        party_counts[party] = Counter(tokens)

    return party_counts

party_counts = get_party_word_counts(
    "../../project/data/clean/Japan_sorted_manifestos.csv",
    start_year=2001,
    end_year=2021
)
party_word_sets = {party: set(counter) for party, counter in party_counts.items()}
common_to_all = set.intersection(*party_word_sets.values())

overlap_df = pd.DataFrame([
    {
        "word": word,
        **{party: party_counts[party][word] for party in party_counts},
        "total": sum(party_counts[party][word] for party in party_counts)
    }
    for word in common_to_all
]).sort_values(by="total", ascending=False).reset_index(drop=True)

overlap_df


,word,Constitutional Democratic Party of Japan,Japan Restoration Party,Japanese Communist Party,Liberal Democratic Party,New Clean Government Party,Party of Hope,Social Democratic Party,total
0,的,13,60,131,125,122,27,88,566
1,化,10,123,58,120,136,28,55,530
2,者,8,42,90,81,126,18,79,444
3,等,3,83,3,117,161,11,31,409
4,法,11,134,77,27,26,11,84,370
...,...,...,...,...,...,...,...,...,...
211,燃料,1,2,5,3,3,1,2,17
212,込ま,1,2,6,1,1,1,2,14
213,現在,1,1,3,1,3,3,1,13
214,実験,1,1,4,2,1,2,1,12


Extraction of nouns 

In [26]:
def extract_noun_chunks(csv_path, start_year=2001, end_year=2021, top_n=20):
    df = pd.read_csv(csv_path)
    df["year"] = pd.to_datetime(df["date"]).dt.year
    df = df[(df["year"] >= start_year) & (df["year"] <= end_year)]

    chunk_counts = Counter()

    for text in df["text"].astype(str):
        for chunk in chunk_text(text):
            doc = nlp(chunk)
            for noun_chunk in doc.noun_chunks:
                chunk_counts[noun_chunk.text] += 1

    return chunk_counts.most_common(top_n)

extract_noun_chunks("../../project/data/clean/Japan_sorted_manifestos.csv")


[('国民', 143),
 ('国', 107),
 ('ため', 105),
 ('日本', 78),
 ('安倍政権', 69),
 ('地域', 66),
 ('大企業', 56),
 ('日本共産党', 51),
 ('実現', 48),
 ('強化', 44),
 ('これ', 44),
 ('女性', 41),
 ('導入', 36),
 ('整備', 36),
 ('推進', 33),
 ('経済', 33),
 ('所得', 31),
 ('充実', 31),
 ('平和', 29),
 ('政治', 28)]

In [27]:
def extract_ngrams(csv_path, n=2, start_year=2001, end_year=2021, top_n=20):
    df = pd.read_csv(csv_path)
    df["year"] = pd.to_datetime(df["date"]).dt.year
    df = df[(df["year"] >= start_year) & (df["year"] <= end_year)]

    ngram_counts = Counter()

    for text in df["text"].astype(str):
        for chunk in chunk_text(text):
            doc = nlp(chunk)
            tokens = [token.text for token in doc if not token.is_space and not token.is_punct]
            for i in range(len(tokens) - n + 1):
                ngram = "".join(tokens[i:i+n])
                ngram_counts[ngram] += 1

    return ngram_counts.most_common(top_n)

bigrams = extract_ngrams("../../project/data/clean/Japan_sorted_manifestos.csv", n=2)
trigrams = extract_ngrams("../../project/data/clean/Japan_sorted_manifestos.csv", n=3)
bigrams, trigrams


([('します', 719),
  ('して', 351),
  ('への', 316),
  ('した', 244),
  ('的な', 198),
  ('てい', 196),
  ('とし', 192),
  ('ともに', 180),
  ('等の', 176),
  ('ととも', 173),
  ('による', 167),
  ('的に', 163),
  ('います', 162),
  ('者の', 132),
  ('を推進', 132),
  ('国民の', 122),
  ('推進し', 122),
  ('するため', 118),
  ('法案', 117),
  ('に取り', 115)],
 [('とともに', 173),
  ('ています', 155),
  ('として', 146),
  ('について', 92),
  ('推進します', 92),
  ('を推進し', 91),
  ('取り組みます', 88),
  ('に取り組み', 83),
  ('における', 81),
  ('日本共産党', 75),
  ('するととも', 75),
  ('を図ります', 74),
  ('強化します', 68),
  ('を強化し', 63),
  ('を実現し', 56),
  ('をすすめます', 54),
  ('に向けた', 54),
  ('してい', 51),
  ('を進めます', 51),
  ('によって', 49)])

Extraction of political terms 

In [28]:
from collections import Counter, defaultdict

def extract_political_phrases(csv_path, n=2, start_year=2001, end_year=2021,
                                min_doc_freq=3, top_n=30):
    df = pd.read_csv(csv_path)
    df["year"] = pd.to_datetime(df["date"]).dt.year
    df = df[(df["year"] >= start_year) & (df["year"] <= end_year)]

    phrase_counts = Counter()
    phrase_doc_ids = defaultdict(set)

    for _, row in df.iterrows():
        text = str(row["text"])
        manifesto_id = row["manifesto_id"]

        for chunk in chunk_text(text):
            doc = nlp(chunk)
            tokens = [token for token in doc if not token.is_space and not token.is_punct]

            i = 0
            while i < len(tokens):
                # try to build a noun-based phrase starting at i
                phrase_tokens = []
                j = i
                while j < len(tokens) and (
                    tokens[j].pos_ in ("NOUN", "PROPN")
                    or (tokens[j].text == "の" and phrase_tokens)
                ):
                    phrase_tokens.append(tokens[j].text)
                    j += 1

                if len(phrase_tokens) >= n:
                    phrase = "".join(phrase_tokens)
                    phrase_counts[phrase] += 1
                    phrase_doc_ids[phrase].add(manifesto_id)

                i = j if j > i else i + 1

    rows = [
        {"phrase": phrase, "total_count": count, "doc_freq": len(phrase_doc_ids[phrase])}
        for phrase, count in phrase_counts.items()
        if len(phrase_doc_ids[phrase]) >= min_doc_freq
    ]

    result_df = pd.DataFrame(rows).sort_values("total_count", ascending=False).head(top_n)
    return result_df.reset_index(drop=True)

political_phrases_df = extract_political_phrases(
    "../../project/data/clean/Japan_sorted_manifestos.csv",
    n=2,
    min_doc_freq=3,
    top_n=30
)
political_phrases_df


,phrase,total_count,doc_freq
0,安倍政権,62,6
1,大企業,43,7
2,消費税,36,4
3,中小企業,28,7
4,兆円,23,3
5,方改革,22,5
6,富裕層,20,3
7,安倍首相,19,3
8,社会保障,18,5
9,国民の所得,15,4


In [33]:
from collections import Counter, defaultdict

def extract_political_phrases(csv_path, n=2, start_year=2001, end_year=2021,
                                min_doc_freq=2, top_n=100):
    df = pd.read_csv(csv_path)
    df["year"] = pd.to_datetime(df["date"]).dt.year
    df = df[(df["year"] >= start_year) & (df["year"] <= end_year)]

    phrase_counts = Counter()
    phrase_doc_ids = defaultdict(set)
    phrase_parties = defaultdict(set)

    for _, row in df.iterrows():
        text = str(row["text"])
        manifesto_id = row["manifesto_id"]
        party = row["partyname"]

        for chunk in chunk_text(text):
            doc = nlp(chunk)
            tokens = [token for token in doc if not token.is_space and not token.is_punct]

            i = 0
            while i < len(tokens):
                phrase_tokens = []
                j = i
                while j < len(tokens) and (
                    tokens[j].pos_ in ("NOUN", "PROPN")
                    or (tokens[j].text == "の" and phrase_tokens)
                ):
                    phrase_tokens.append(tokens[j].text)
                    j += 1

                if len(phrase_tokens) >= n:
                    phrase = "".join(phrase_tokens)
                    phrase_counts[phrase] += 1
                    phrase_doc_ids[phrase].add(manifesto_id)
                    phrase_parties[phrase].add(party)

                i = j if j > i else i + 1

    rows = [
        {
            "phrase": phrase,
            "total_count": count,
            "doc_freq": len(phrase_doc_ids[phrase]),
            "party_count": len(phrase_parties[phrase]),
            "parties": ", ".join(sorted(phrase_parties[phrase]))
        }
        for phrase, count in phrase_counts.items()
        if len(phrase_doc_ids[phrase]) >= min_doc_freq
    ]

    result_df = pd.DataFrame(rows).sort_values("total_count", ascending=False).head(top_n)
    return result_df.reset_index(drop=True)

political_phrases_df = extract_political_phrases(
    "../../project/data/clean/Japan_sorted_manifestos.csv",
    n=2,
    min_doc_freq=2,
    top_n=100
)
political_phrases_df


,phrase,total_count,doc_freq,party_count,parties
0,安倍政権,62,6,4,"Constitutional Democratic Party of Japan, Japa..."
1,日本共産党,50,2,1,Japanese Communist Party
2,大企業,43,7,5,"Japan Restoration Party, Japanese Communist Pa..."
3,消費税,36,4,3,"Japanese Communist Party, Liberal Democratic P..."
4,中小企業,28,7,5,"Japanese Communist Party, Liberal Democratic P..."
...,...,...,...,...,...
95,支援体制,5,4,3,"Liberal Democratic Party, New Clean Government..."
96,高齢化,5,4,4,"Japan Restoration Party, Japanese Communist Pa..."
97,つの改革,5,2,1,Japanese Communist Party
98,支援策,5,3,2,"New Clean Government Party, Social Democratic ..."


In [35]:
from collections import Counter, defaultdict

# POS tags that count as "content" and can extend a phrase
CONTENT_POS = {"NOUN", "PROPN", "VERB", "ADJ", "NUM"}

def extract_full_phrases(csv_path, n=2, start_year=2001, end_year=2021,
                          min_doc_freq=2, top_n=100):
    df = pd.read_csv(csv_path)
    df["year"] = pd.to_datetime(df["date"]).dt.year
    df = df[(df["year"] >= start_year) & (df["year"] <= end_year)]

    phrase_counts = Counter()
    phrase_doc_ids = defaultdict(set)
    phrase_parties = defaultdict(set)

    for _, row in df.iterrows():
        text = str(row["text"])
        manifesto_id = row["manifesto_id"]
        party = row["partyname"]

        for chunk in chunk_text(text):
            doc = nlp(chunk)
            tokens = [token for token in doc if not token.is_space and not token.is_punct]

            i = 0
            while i < len(tokens):
                phrase_tokens = []
                j = i
                while j < len(tokens) and (
                    tokens[j].pos_ in CONTENT_POS
                    or (tokens[j].text == "の" and phrase_tokens)
                ):
                    phrase_tokens.append(tokens[j].text)
                    j += 1

                if len(phrase_tokens) >= n:
                    phrase = "".join(phrase_tokens)
                    phrase_counts[phrase] += 1
                    phrase_doc_ids[phrase].add(manifesto_id)
                    phrase_parties[phrase].add(party)

                i = j if j > i else i + 1

    rows = [
        {
            "phrase": phrase,
            "total_count": count,
            "doc_freq": len(phrase_doc_ids[phrase]),
            "party_count": len(phrase_parties[phrase]),
            "parties": ", ".join(sorted(phrase_parties[phrase]))
        }
        for phrase, count in phrase_counts.items()
        if len(phrase_doc_ids[phrase]) >= min_doc_freq
    ]

    result_df = pd.DataFrame(rows).sort_values("total_count", ascending=False).head(top_n)
    return result_df.reset_index(drop=True)

full_phrases_df = extract_full_phrases(
    "../../project/data/clean/Japan_sorted_manifestos.csv",
    n=2,
    min_doc_freq=2,
    top_n=100
)
full_phrases_df


,phrase,total_count,doc_freq,party_count,parties
0,取り組み,96,7,5,"Japanese Communist Party, Liberal Democratic P..."
1,安倍政権,57,6,4,"Constitutional Democratic Party of Japan, Japa..."
2,日本共産党,47,2,1,Japanese Communist Party
3,大企業,41,7,5,"Japan Restoration Party, Japanese Communist Pa..."
4,消費税,20,2,1,Japanese Communist Party
...,...,...,...,...,...
95,企業団体献金,5,3,2,"Japanese Communist Party, Social Democratic Party"
96,2030年,5,2,2,"Japanese Communist Party, Party of Hope"
97,盛り込ま,4,3,2,"Japan Restoration Party, Japanese Communist Party"
98,障がい者,4,2,2,"Japan Restoration Party, New Clean Government ..."


In [38]:
from collections import Counter, defaultdict

NOUN_POS = {"NOUN", "PROPN", "PRON"}

def extract_pure_noun_phrases(csv_path, n=2, start_year=2001, end_year=2021,
                               min_doc_freq=2, top_n=100):
    df = pd.read_csv(csv_path)
    df["year"] = pd.to_datetime(df["date"]).dt.year
    df = df[(df["year"] >= start_year) & (df["year"] <= end_year)]

    phrase_counts = Counter()
    phrase_doc_ids = defaultdict(set)
    phrase_parties = defaultdict(set)

    for _, row in df.iterrows():
        text = str(row["text"])
        manifesto_id = row["manifesto_id"]
        party = row["partyname"]

        for chunk in chunk_text(text):
            doc = nlp(chunk)
            tokens = [token for token in doc if not token.is_space and not token.is_punct]

            i = 0
            while i < len(tokens):
                phrase_tokens = []
                j = i
                while j < len(tokens) and (
                    tokens[j].pos_ in NOUN_POS
                    or (tokens[j].text == "の" and phrase_tokens)
                ):
                    phrase_tokens.append(tokens[j].text)
                    j += 1

                if len(phrase_tokens) >= n:
                    phrase = "".join(phrase_tokens)
                    phrase_counts[phrase] += 1
                    phrase_doc_ids[phrase].add(manifesto_id)
                    phrase_parties[phrase].add(party)

                i = j if j > i else i + 1

    rows = [
        {
            "phrase": phrase,
            "total_count": count,
            "doc_freq": len(phrase_doc_ids[phrase]),
            "party_count": len(phrase_parties[phrase]),
            "parties": ", ".join(sorted(phrase_parties[phrase]))
        }
        for phrase, count in phrase_counts.items()
        if len(phrase_doc_ids[phrase]) >= min_doc_freq
    ]

    result_df = pd.DataFrame(rows).sort_values("total_count", ascending=False).head(top_n)
    return result_df.reset_index(drop=True)

pure_noun_phrases_df = extract_pure_noun_phrases(
    "../../project/data/clean/Japan_sorted_manifestos.csv",
    n=2,
    min_doc_freq=2,
    top_n=100
)
pure_noun_phrases_df


,phrase,total_count,doc_freq,party_count,parties
0,安倍政権,62,6,4,"Constitutional Democratic Party of Japan, Japa..."
1,日本共産党,50,2,1,Japanese Communist Party
2,大企業,43,7,5,"Japan Restoration Party, Japanese Communist Pa..."
3,消費税,36,4,3,"Japanese Communist Party, Liberal Democratic P..."
4,中小企業,28,7,5,"Japanese Communist Party, Liberal Democratic P..."
...,...,...,...,...,...
95,国民の生命,5,2,2,"Liberal Democratic Party, Party of Hope"
96,歳児,5,3,3,"Japan Restoration Party, Liberal Democratic Pa..."
97,高齢化,5,4,4,"Japan Restoration Party, Japanese Communist Pa..."
98,分の,5,3,3,"Japanese Communist Party, Liberal Democratic P..."


Filtering from political terms: nouns, proper nouns, 

In [37]:
from collections import Counter, defaultdict

NOUN_POS = {"NOUN", "PROPN", "PRON"}
KANJI_NUMERALS = {"一", "二", "三", "四", "五", "六", "七", "八", "九", "十", "百", "千", "万", "億", "兆"}

def extract_pure_noun_phrases(csv_path, n=2, start_year=2001, end_year=2021,
                               min_doc_freq=2, top_n=100):
    df = pd.read_csv(csv_path)
    df["year"] = pd.to_datetime(df["date"]).dt.year
    df = df[(df["year"] >= start_year) & (df["year"] <= end_year)]

    phrase_counts = Counter()
    phrase_doc_ids = defaultdict(set)
    phrase_parties = defaultdict(set)

    for _, row in df.iterrows():
        text = str(row["text"])
        manifesto_id = row["manifesto_id"]
        party = row["partyname"]

        for chunk in chunk_text(text):
            doc = nlp(chunk)
            tokens = [token for token in doc if not token.is_space and not token.is_punct]

            i = 0
            while i < len(tokens):
                phrase_tokens = []
                j = i
                while j < len(tokens) and (
                    tokens[j].pos_ in NOUN_POS
                    or (tokens[j].text == "の" and phrase_tokens)
                    or (tokens[j].pos_ == "NUM" and tokens[j].text in KANJI_NUMERALS and phrase_tokens)
                ):
                    phrase_tokens.append(tokens[j].text)
                    j += 1

                if len(phrase_tokens) >= n:
                    phrase = "".join(phrase_tokens)
                    phrase_counts[phrase] += 1
                    phrase_doc_ids[phrase].add(manifesto_id)
                    phrase_parties[phrase].add(party)

                i = j if j > i else i + 1

    rows = [
        {
            "phrase": phrase,
            "total_count": count,
            "doc_freq": len(phrase_doc_ids[phrase]),
            "party_count": len(phrase_parties[phrase]),
            "parties": ", ".join(sorted(phrase_parties[phrase]))
        }
        for phrase, count in phrase_counts.items()
        if len(phrase_doc_ids[phrase]) >= min_doc_freq
    ]

    result_df = pd.DataFrame(rows).sort_values("total_count", ascending=False).head(top_n)
    return result_df.reset_index(drop=True)

pure_noun_phrases_df = extract_pure_noun_phrases(
    "../../project/data/clean/Japan_sorted_manifestos.csv",
    n=2,
    min_doc_freq=2,
    top_n=100
)
pure_noun_phrases_df


,phrase,total_count,doc_freq,party_count,parties
0,安倍政権,62,6,4,"Constitutional Democratic Party of Japan, Japa..."
1,日本共産党,50,2,1,Japanese Communist Party
2,大企業,43,7,5,"Japan Restoration Party, Japanese Communist Pa..."
3,消費税,36,4,3,"Japanese Communist Party, Liberal Democratic P..."
4,中小企業,28,7,5,"Japanese Communist Party, Liberal Democratic P..."
...,...,...,...,...,...
95,財政収支,5,3,3,"Japanese Communist Party, Liberal Democratic P..."
96,障がい者,5,2,2,"Japan Restoration Party, New Clean Government ..."
97,分の,5,3,3,"Japanese Communist Party, Liberal Democratic P..."
98,条の精神,5,2,1,Japanese Communist Party
